In [1]:
import pandas as pd

from pathlib import Path

PAPER_PATH = Path("../papers/")

In [73]:
PAPER_PATH / "parsed_papers.parquet"

WindowsPath('../papers/parsed_papers.parquet')

In [2]:
df = pd.read_parquet(PAPER_PATH / "parsed_papers.parquet")
df.head(10)

,paper_path,paper_id,page_number,text
0,papers\2607.00283v1.pdf,2607.00283v1,0,# **What’s Hidden Matters: Identifying Plannin...
1,papers\2607.00283v1.pdf,2607.00283v1,1,\n\n<!-- Start of picture text -->\nnuScenes +...
2,papers\2607.00283v1.pdf,2607.00283v1,2,"driving stack as a causal graph, as in DriveLM..."
3,papers\2607.00283v1.pdf,2607.00283v1,3,_3) Structured Annotation Generation:_ The key...
4,papers\2607.00283v1.pdf,2607.00283v1,4,as described in Section III-C.3. Our final dat...
5,papers\2607.00283v1.pdf,2607.00283v1,5,TABLE II: Supervised fine-tuned model performa...
6,papers\2607.00283v1.pdf,2607.00283v1,6,InternVL3.5-8B shows mixed results: accuracy a...
7,papers\2607.00283v1.pdf,2607.00283v1,7,all model families and scales. Our results sho...
8,papers\2607.00283v1.pdf,2607.00283v1,8,"- [36] Zhenhua Xu, Yujia Zhang, Enze Xie, Zhen..."
9,papers\2607.00292v1.pdf,2607.00292v1,0,# An LLM-Based Framework for Intent-Driven Net...


In [3]:
def combine_pages(pages: pd.DataFrame):

    return pd.Series({
        "paper_id": pages["paper_id"].iloc[0],
        "text": "\n".join(pages.sort_values("page_number")["text"]),
        "num_pages": len(pages)
    })

document_df = df.groupby("paper_path").apply(combine_pages, include_groups=False).reset_index()
document_df = document_df[document_df['num_pages'] <= 200].copy() # Remove books and dissertations
document_df.head()

,paper_path,paper_id,text,num_pages
0,papers\2607.00283v1.pdf,2607.00283v1,# **What’s Hidden Matters: Identifying Plannin...,9
1,papers\2607.00292v1.pdf,2607.00292v1,# An LLM-Based Framework for Intent-Driven Net...,9
2,papers\2607.00296v1.pdf,2607.00296v1,# **Learning When to Listen: Gated Affect Fusi...,7
3,papers\2607.00304v1.pdf,2607.00304v1,# Mapping the Evaluation Frontier: An Empirica...,5
4,papers\2607.00310v1.pdf,2607.00310v1,## **RetailSMV: Exocentric vs. Egocentric Adap...,31


We want to split by markdown heading. In total, there are 6 markdown header levels (#, ##, etc.).

We suspect that headers 1-3 are commonly used but 4-6 may be sparingly used.

In [4]:
import re

def count_headings_by_level(text):
    if not isinstance(text, str):
        return {}
    counts = {}
    for level in range(1, 7):
        pattern = rf'^{"#" * level}(?!#)\s'
        counts[f'h{level}'] = len(re.findall(pattern, text, flags=re.MULTILINE))
    return counts

In [5]:

header_counts_df = document_df['text'].apply(count_headings_by_level).apply(pd.Series)


In [6]:
(header_counts_df.sum() / header_counts_df.sum().sum() * 100).round()

h1     9.0
h2    45.0
h3    34.0
h4     8.0
h5     2.0
h6     1.0
dtype: float64

In [1]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

headers_to_split_on = [
    ("#", "h1"),
    ("##", "h2"),
    ("###", "h3"),
    ("####", "h4")
]

md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on, strip_headers=False)
size_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150, length_function=len)

In [8]:
records = []
for row in document_df.itertuples(index=False):
    chunks = size_splitter.split_text(row.text)
    for i, chunk in enumerate(chunks):
        records.append({
            "paper_id": row.paper_id,
            "paper_path": row.paper_path,
            "num_pages": row.num_pages,
            "chunk_id": i,
            "chunk_text": chunk,
            "chunk_char_count": len(chunk),
        })

size_chunks_df = pd.DataFrame.from_records(records)

In [13]:
size_chunks_df["chunk_char_count"].describe(percentiles=[.01,.05,.1,.25,.5,.75,.9])

count    276935.000000
mean       1171.910954
std         334.377195
min           1.000000
1%          102.000000
5%          401.000000
10%         678.000000
25%        1033.000000
50%        1288.000000
75%        1421.000000
90%        1478.000000
max        1500.000000
Name: chunk_char_count, dtype: float64

In [14]:
size_chunks_df

,paper_id,paper_path,num_pages,chunk_id,chunk_text,chunk_char_count
0,2607.00283v1,papers\2607.00283v1.pdf,9,0,# **What’s Hidden Matters: Identifying Plannin...,295
1,2607.00283v1,papers\2607.00283v1.pdf,9,1,**_Abstract_ — Autonomous vehicles must safely...,1499
2,2607.00283v1,papers\2607.00283v1.pdf,9,2,"counterparts, and that our PKL-guided data sel...",324
3,2607.00283v1,papers\2607.00283v1.pdf,9,3,## I. INTRODUCTION \n\nSafe navigation require...,830
4,2607.00283v1,papers\2607.00283v1.pdf,9,4,KL-divergence (PKL) were introduced to quantif...,1360
...,...,...,...,...,...,...
276930,2608.02650v1,papers\2608.02650v1.pdf,10,44,"Antelmi, A.; Cordasco, G.; Polato, M.; Scarano...",1335
276931,2608.02650v1,papers\2608.02650v1.pdf,10,45,"Erdogan, L. E.; Furuta, H.; Kim, S.; Lee, N.; ...",1325
276932,2608.02650v1,papers\2608.02650v1.pdf,10,46,OpenAI. 2023. GPT-4 Technical Report. \n\nPara...,1410
276933,2608.02650v1,papers\2608.02650v1.pdf,10,47,"Schulman, J.; Wolski, F.; Dhariwal, P.; Radfor...",1461


In [50]:
size_chunks_df.sort_values(by="chunk_char_count", ascending=True)[["chunk_text", "chunk_char_count"]].head(10000).to_csv("size_chunks.csv")

After manual review -- we seem to be getting a lot of chunks of size 1 or 2 chars that are just the page numbers. Let's remove all chunks that are 3 characters or smaller to remove some noise. Additionally, PyMuPDF4LLM includes placeholders for images that we need to remove as well.

## Markdown Header Splitter

In [74]:

records = []
for row in document_df.itertuples(index=False):
    header_docs = md_splitter.split_text(row.text)
    chunk_id = 0
    for header_doc in header_docs:
        sub_chunks = md_splitter.split_text(header_doc.page_content)
        for sub_chunk in sub_chunks:
            records.append({
                "paper_id": row.paper_id,
                "paper_path": row.paper_path,
                "chunk_id": chunk_id,
                "h1": header_doc.metadata.get("h1"),
                "h2": header_doc.metadata.get("h2"),
                "h3": header_doc.metadata.get("h3"),
                "h4": header_doc.metadata.get("h4"),
                "chunk_text": sub_chunk.page_content,
                "chunk_char_count": len(sub_chunk.page_content),
                "chunk_word_count": len(sub_chunk.page_content.split())
            })
            chunk_id += 1

md_chunks_df = pd.DataFrame.from_records(records)

In [75]:
md_chunks_df[["chunk_char_count", "chunk_word_count"]].describe(percentiles=[.01,.05,.1,.25,.5,.75,.9])

,chunk_char_count,chunk_word_count
count,125315.000000,125315.000000
mean,2513.303020,345.266289
std,4037.814552,494.644353
min,1.000000,1.000000
1%,19.000000,3.000000
5%,121.000000,15.000000
10%,283.000000,37.000000
25%,711.000000,99.000000
50%,1557.000000,215.000000
75%,3095.000000,425.000000


In [78]:
md_chunks_df.sort_values(["chunk_char_count", "chunk_word_count"], ascending=True).head(1000).to_csv("md_chunks.csv")

In [77]:
md_chunks_df["chunk_char_count"].sum()

np.int64(314954568)

In [93]:
print(md_chunks_df[md_chunks_df["paper_id"]=="2607.16930v1"]["chunk_text"].tolist()[-3])

## _C. Theoretical Complexity and Real-Time Feasibility_
The empirical latency measurements are consistent with theoretical complexity expectations. Random Forest evaluates all _M_ trees to depth _d_ for every prediction, giving _O_ ( _M · d_ ) inference complexity with a large constant due to memoryscattered tree traversal. HGBR and XGBoost share the same _O_ ( _T · B_ ) asymptotic complexity per boosting round _T_ over _B_ histogram bins; however, HGBR’s contiguous bin table layout and XGBoost’s cache-aware block structure reduce the memory access constant significantly. This explains why HGBR achieves the lowest universal inference latency (0.00246 ms) and the absolute minimum at the micro-agent level (0.00122 ms for the Canopy expert), while XGBoost achieves the shortest training time (0.61 s universal) due to its approximate greedy splitting.
